# Tests de integración de `fix_trips_correspondence()`

Notebook orientado a verificar el comportamiento end-to-end de OP-03 `fix trips correspondence`,
considerando resultado tabular, resultado operacional y trazabilidad.

## Sección 0. Preparación

Esta sección deja lista la infraestructura mínima del notebook:
- imports generales,
- imports del módulo,
- helpers de testing reutilizables,
- y configuración básica de display.

### 0.1 Imports generales

In [24]:
import json
from pprint import pprint
from copy import deepcopy
import pandas as pd
import numpy as np

### 0.2 Imports del módulo 

In [2]:
from pylondrina.schema import DomainSpec, FieldSpec, TripSchema, TripSchemaEffective
from pylondrina.datasets import TripDataset
from pylondrina.reports import OperationReport
from pylondrina.errors import FixError
from pylondrina.fixing import FixCorrespondenceOptions, fix_trips_correspondence

### 0.3 Helpers de testing reutilizables

In [3]:
def show_ok(label: str) -> None:
    print(f"OK - {label}")


def issue_codes(report_or_issues):
    issues = report_or_issues.issues if hasattr(report_or_issues, "issues") else report_or_issues
    return [i.code for i in issues]


def assert_issue_present(report_or_issues, code: str) -> None:
    codes = issue_codes(report_or_issues)
    assert code in codes, f"No se encontró {code}. Codes actuales: {codes}"


def assert_issue_absent(report_or_issues, code: str) -> None:
    codes = issue_codes(report_or_issues)
    assert code not in codes, f"Se encontró inesperadamente {code}. Codes actuales: {codes}"


def make_field(
    name: str,
    dtype: str,
    *,
    required: bool = False,
    constraints: dict | None = None,
    domain: DomainSpec | None = None,
) -> FieldSpec:
    return FieldSpec(
        name=name,
        dtype=dtype,
        required=required,
        constraints=constraints,
        domain=domain,
    )


BASE_FIX_SCHEMA = TripSchema(
    version="1.1",
    fields={
        "movement_id": make_field("movement_id", "string", required=True),
        "user_id": make_field("user_id", "string", required=True),
        "mode": make_field(
            "mode",
            "categorical",
            domain=DomainSpec(
                values=["walk", "bus", "metro", "car", "unknown"],
                extendable=True,
            ),
        ),
        "purpose": make_field(
            "purpose",
            "categorical",
            domain=DomainSpec(
                values=["work", "study", "shopping", "health", "unknown"],
                extendable=True,
            ),
        ),
        "trip_weight": make_field("trip_weight", "float"),
    },
    required=["movement_id", "user_id"],
    semantic_rules=None,
)


def _base_domains_effective(schema: TripSchema, fields_present: list[str]) -> dict:
    out = {}
    for field_name in fields_present:
        fs = schema.fields.get(field_name)
        if fs is None or fs.domain is None:
            continue
        values = sorted(list(fs.domain.values))
        out[field_name] = {
            "values": values,
            "extended": False,
            "added_values": [],
            "unknown_value": "unknown" if "unknown" in values else None,
            "strict_applied": False,
        }
    return out


def make_tripdataset_fixture(
    df: pd.DataFrame,
    *,
    schema: TripSchema | None = None,
    dataset_id: str = "ds-op03-it-001",
    is_validated: bool = True,
    field_correspondence: dict | None = None,
    value_correspondence: dict | None = None,
    events: list[dict] | None = None,
) -> TripDataset:
    schema = schema or BASE_FIX_SCHEMA

    if field_correspondence is None:
        field_correspondence = {c: c for c in df.columns if c in schema.fields}

    if value_correspondence is None:
        value_correspondence = {}

    base_domains = _base_domains_effective(
        schema=schema,
        fields_present=[c for c in df.columns if c in schema.fields],
    )

    metadata = {
        "dataset_id": dataset_id,
        "events": deepcopy(events) if events is not None else [],
        "is_validated": is_validated,
        "mappings": {
            "field_correspondence": deepcopy(field_correspondence),
            "value_correspondence": deepcopy(value_correspondence),
        },
        "domains_effective": deepcopy(base_domains),
    }

    schema_effective = TripSchemaEffective(
        dtype_effective={},
        overrides={},
        domains_effective=deepcopy(base_domains),
        temporal={},
        fields_effective=list(df.columns),
    )

    provenance = {
        "source": {
            "name": "synthetic",
            "entity": "trips",
            "version": "op03-integration-tests-v1",
        },
        "notes": ["fixture manual para tests de integración de OP-03"],
    }

    return TripDataset(
        data=df.copy(deep=True),
        schema=schema,
        schema_version=schema.version,
        provenance=provenance,
        field_correspondence=deepcopy(field_correspondence),
        value_correspondence=deepcopy(value_correspondence),
        metadata=metadata,
        schema_effective=schema_effective,
    )

### 0.4 Configuración de display

In [4]:
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

## Sección 1. Fixtures base de OP-03

### Fixture 1 - tripdataset_canonical_small

In [5]:
def tripdataset_canonical_small() -> TripDataset:
    df = pd.DataFrame({
        "movement_id": ["m1", "m2"],
        "user_id": ["u1", "u2"],
        "mode": ["bus", "walk"],
        "purpose": ["work", "study"],
        "trip_weight": [1.0, 2.0],
    })
    return make_tripdataset_fixture(
        df,
        dataset_id="tripdataset_canonical_small",
        is_validated=True,
    )

### Fixture 2 - tripdataset_with_categories_to_fix

In [6]:
def tripdataset_with_categories_to_fix() -> TripDataset:
    df = pd.DataFrame({
        "movement_id": ["m1", "m2", "m3"],
        "user_id": ["u1", "u2", "u3"],
        "modo_raw": ["BUS", "WALK", "BUS"],
        "purpose_raw": ["WORK", "STUDY", "WORK"],
        "trip_weight": [1.0, 2.0, 3.0],
    })
    return make_tripdataset_fixture(
        df,
        dataset_id="tripdataset_with_categories_to_fix",
        is_validated=True,
        field_correspondence={
            "movement_id": "movement_id",
            "user_id": "user_id",
        },
    )

### Fixture 3 - tripdataset_unvalidated_small

In [7]:
def tripdataset_unvalidated_small() -> TripDataset:
    df = pd.DataFrame({
        "movement_id": ["m1", "m2"],
        "user_id": ["u1", "u2"],
        "mode": ["BUS", "WALK"],
        "purpose": ["WORK", "STUDY"],
        "trip_weight": [1.0, 2.0],
    })
    prior_events = [
        {
            "op": "import_trips",
            "ts_utc": "2026-04-02T20:00:00Z",
            "parameters": {"source_name": "synthetic"},
            "summary": {"rows_in": 2, "rows_out": 2},
            "issues_summary": {"counts": {"info": 0, "warning": 0, "error": 0}},
        }
    ]
    return make_tripdataset_fixture(
        df,
        dataset_id="tripdataset_unvalidated_small",
        is_validated=False,
        events=prior_events,
    )

## Sección 2. Tests de integración

### Test IT-01: Caso principal correcto: aplica fields -> values, actualiza mappings/domains y no muta el input

In [11]:
trips = tripdataset_with_categories_to_fix()
input_data_before = trips.data.copy(deep=True)
input_metadata_before = deepcopy(trips.metadata)

fixed, report = fix_trips_correspondence(
    trips,
    field_corrections={
        "modo_raw": "mode",
        "purpose_raw": "purpose",
    },
    value_corrections={
        "mode": {"BUS": "bus", "WALK": "walk"},
        "purpose": {"WORK": "work", "STUDY": "study"},
    },
    options=FixCorrespondenceOptions(),
    correspondence_context={
        "reason": "normalización manual mínima",
        "notes": "caso principal correcto OP-03",
    },
)

# Retornos
assert isinstance(fixed, TripDataset)
assert isinstance(report, OperationReport)
assert report.ok is True

# No mutación del input
pd.testing.assert_frame_equal(trips.data, input_data_before)
assert trips.metadata == input_metadata_before

# Resultado tabular
assert list(fixed.data.columns) == ["movement_id", "user_id", "mode", "purpose", "trip_weight"]
assert fixed.data["mode"].tolist() == ["bus", "walk", "bus"]
assert fixed.data["purpose"].tolist() == ["work", "study", "work"]
assert len(fixed.data) == len(trips.data)

# Summary mínimo
assert report.summary["n_rows"] == 3
assert report.summary["n_field_corrections_requested"] == 2
assert report.summary["n_field_corrections_applied"] == 2
assert report.summary["n_value_corrections_fields_requested"] == 2
assert report.summary["n_value_corrections_fields_applied"] == 2
assert report.summary["n_value_replacements_applied"] == 6
assert report.summary["noop"] is False
assert set(report.summary["domains_effective_updated_fields"]) == {"mode", "purpose"}

# Estado lógico
assert fixed.metadata["is_validated"] is False
assert fixed.metadata["dataset_id"] == trips.metadata["dataset_id"]
assert fixed.provenance == trips.provenance

# Correspondencias efectivas finales
assert fixed.field_correspondence["mode"] == "modo_raw"
assert fixed.field_correspondence["purpose"] == "purpose_raw"
assert fixed.value_correspondence["mode"]["BUS"] == "bus"
assert fixed.value_correspondence["purpose"]["WORK"] == "work"

# metadata["mappings"] coherente
assert fixed.metadata["mappings"]["field_correspondence"] == fixed.field_correspondence
assert fixed.metadata["mappings"]["value_correspondence"] == fixed.value_correspondence

# domains_effective actualizado solo donde corresponde
assert set(fixed.metadata["domains_effective"]["mode"]["values"]) == {"bus", "walk"}
assert set(fixed.metadata["domains_effective"]["purpose"]["values"]) == {"work", "study"}
assert set(fixed.schema_effective.domains_effective["mode"]["values"]) == {"bus", "walk"}
assert set(fixed.schema_effective.domains_effective["purpose"]["values"]) == {"work", "study"}

# schema_effective mínimo
assert fixed.schema_effective.dtype_effective == trips.schema_effective.dtype_effective
assert fixed.schema_effective.overrides == trips.schema_effective.overrides
assert fixed.schema_effective.temporal == trips.schema_effective.temporal
assert set(fixed.schema_effective.fields_effective) == set(["movement_id", "user_id", "mode", "purpose", "trip_weight"])

# Evento mínimo
assert len(fixed.metadata["events"]) == 1
event = fixed.metadata["events"][-1]
assert event["op"] == "fix_trips_correspondence"
assert event["summary"] == report.summary
assert event["parameters"] == report.parameters
assert "issues_summary" in event
assert event["context"]["reason"] == "normalización manual mínima"

display(trips.data)
display(fixed.data)
display(trips.metadata["events"])
display(report.issues)
show_ok("IT-01 - caso principal correcto")

,movement_id,user_id,modo_raw,purpose_raw,trip_weight
0,m1,u1,BUS,WORK,1.0
1,m2,u2,WALK,STUDY,2.0
2,m3,u3,BUS,WORK,3.0


,movement_id,user_id,mode,purpose,trip_weight
0,m1,u1,bus,work,1.0
1,m2,u2,walk,study,2.0
2,m3,u3,bus,work,3.0


[]

[Issue(level='info', code='FIX.DOMAINS.UPDATED', message='Se actualizaron los domains_effective para los campos tocados por value_corrections.', field=None, source_field=None, row_count=None, details={'kind': 'domains_effective', 'updated_fields': ['mode', 'purpose'], 'added_values_by_field': {}, 'n_rows_total': 3}),
 Issue(level='info', code='FIX.INFO.FIELD_CORRECTIONS_APPLIED', message='Se aplicaron correcciones de campos (requested=2, applied=2).', field=None, source_field=None, row_count=None, details={'kind': 'field_corrections', 'requested_count': 2, 'applied_count': 2, 'mapping_sample': {'modo_raw': 'mode', 'purpose_raw': 'purpose'}, 'n_rows_total': 3}),
 Issue(level='info', code='FIX.INFO.VALUE_CORRECTIONS_APPLIED', message='Se aplicaron correcciones de valores (requested_fields=2, applied_fields=2, replacements=6).', field=None, source_field=None, row_count=None, details={'kind': 'value_corrections', 'requested_fields_count': 2, 'applied_fields_count': 2, 'replacements_count':

OK - IT-01 - caso principal correcto


### Test IT-02: Caso fatal/precondición: correspondence_context root inválido aborta sin side effects

In [12]:
trips = tripdataset_canonical_small()
input_data_before = trips.data.copy(deep=True)
input_metadata_before = deepcopy(trips.metadata)

raised = None
try:
    fix_trips_correspondence(
        trips,
        field_corrections=None,
        value_corrections=None,
        options=FixCorrespondenceOptions(),
        correspondence_context=["not", "a", "dict"],  # root inválido
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, FixError)

# El input debe quedar intacto
pd.testing.assert_frame_equal(trips.data, input_data_before)
assert trips.metadata == input_metadata_before
assert trips.metadata["events"] == []

show_ok("IT-02 - fatal de precondición")

OK - IT-02 - fatal de precondición


### Test IT-03: Caso degradado con warning: aplicación parcial de fields/values y contexto saneado

In [18]:
trips = tripdataset_with_categories_to_fix()
input_data_before = trips.data.copy(deep=True)

fixed, report = fix_trips_correspondence(
    trips,
    field_corrections={
        "purpose_raw": "purpose",
        "missing_mode": "mode",  # source faltante
    },
    value_corrections={
        "purpose": {
            "WORK": "work",
            "MISSING_VALUE": "health",  # source value ausente
        }
    },
    options=FixCorrespondenceOptions(strict=False),
    correspondence_context={
        "reason": "ajuste parcial",
        "unknown_key": "debe descartarse",
        "notes": {"ok": "texto", "bad": object()},
    },
)
display(trips.data)
display(fixed.data)

# El input no se muta
pd.testing.assert_frame_equal(trips.data, input_data_before)
# Sí hubo cambio efectivo parcial
assert "purpose" in fixed.data.columns
assert fixed.data["purpose"].tolist() == ["work", "STUDY", "work"]
assert fixed.metadata["is_validated"] is False
assert report.summary["noop"] is False

# Warnings / degradación
assert_issue_present(report, "FIX.FIELD.SOURCE_COLUMN_MISSING")
assert_issue_present(report, "FIX.FIELD.PARTIAL_APPLY")
assert_issue_present(report, "FIX.VALUE.SOURCE_VALUES_NOT_FOUND")
assert_issue_present(report, "FIX.CONTEXT.UNKNOWN_KEYS_DROPPED")
assert_issue_present(report, "FIX.CONTEXT.NON_SERIALIZABLE_DROPPED")

# Conteos coherentes
assert report.summary["n_field_corrections_requested"] == 2
assert report.summary["n_field_corrections_applied"] == 1
assert report.summary["n_value_corrections_fields_requested"] == 1
assert report.summary["n_value_corrections_fields_applied"] == 1
assert report.summary["n_value_replacements_applied"] == 2

# Context saneado en evento
event = fixed.metadata["events"][-1]
assert "unknown_key" not in event["context"]
assert event["context"]["notes"]["ok"] == "texto"
assert "bad" not in event["context"]["notes"]

show_ok("IT-03 - degradado con warning")

,movement_id,user_id,modo_raw,purpose_raw,trip_weight
0,m1,u1,BUS,WORK,1.0
1,m2,u2,WALK,STUDY,2.0
2,m3,u3,BUS,WORK,3.0


,movement_id,user_id,modo_raw,purpose,trip_weight
0,m1,u1,BUS,work,1.0
1,m2,u2,WALK,STUDY,2.0
2,m3,u3,BUS,work,3.0


OK - IT-03 - degradado con warning


### Test IT-04: Caso metadata/evento/summary: append-only, preservación de identidad y coherencia report-event

In [19]:
trips = tripdataset_unvalidated_small()
prior_events_before = deepcopy(trips.metadata["events"])
dataset_id_before = trips.metadata["dataset_id"]
provenance_before = deepcopy(trips.provenance)

fixed, report = fix_trips_correspondence(
    trips,
    field_corrections=None,
    value_corrections={
        "mode": {"BUS": "bus", "WALK": "walk"},
        "purpose": {"WORK": "work", "STUDY": "study"},
    },
    options=FixCorrespondenceOptions(),
    correspondence_context={"reason": "normalización categórica"},
)

# Nuevo dataset
assert fixed is not trips

# Preservación de identidad y provenance
assert fixed.metadata["dataset_id"] == dataset_id_before
assert fixed.provenance == provenance_before
assert "artifact_id" not in fixed.metadata

# Append-only de eventos
assert len(trips.metadata["events"]) == len(prior_events_before)
assert len(fixed.metadata["events"]) == len(prior_events_before) + 1
assert fixed.metadata["events"][:-1] == prior_events_before

# Evento nuevo al final
event = fixed.metadata["events"][-1]
assert event["op"] == "fix_trips_correspondence"
assert event["summary"] == report.summary
assert event["parameters"] == report.parameters
assert "issues_summary" in event
assert "context" in event

# Report y evento alineados
assert report.summary["n_rows"] == len(fixed.data)
assert report.summary["noop"] is False
assert report.parameters["strict"] is False
assert report.parameters["field_corrections"] is None
assert "value_corrections" in report.parameters

# Como ya venía no validado y hubo cambio, sigue False
assert fixed.metadata["is_validated"] is False

show_ok("IT-04 - metadata/evento/summary")

OK - IT-04 - metadata/evento/summary


### Test IT-05: Caso de política clave: sin cambios efectivos preserva is_validated y deja evidencia

In [21]:
trips = tripdataset_canonical_small()
input_data_before = trips.data.copy(deep=True)
validated_before = trips.metadata["is_validated"]

fixed, report = fix_trips_correspondence(
    trips,
    field_corrections=None,
    value_corrections=None,
    options=FixCorrespondenceOptions(),
)

# No mutación del input
pd.testing.assert_frame_equal(trips.data, input_data_before)

# Dataset resultante equivalente
pd.testing.assert_frame_equal(fixed.data, trips.data)

# Política NO_EFFECTIVE_CHANGES / sin cambios efectivos
assert report.summary["noop"] is True
assert_issue_present(report, "FIX.NO_EFFECTIVE_CHANGES.NO_CORRECTIONS")
assert fixed.metadata["is_validated"] == validated_before

# Igual deja evidencia
assert len(fixed.metadata["events"]) == 1
event = fixed.metadata["events"][-1]
assert event["op"] == "fix_trips_correspondence"
assert event["summary"] == report.summary
assert event["parameters"] == report.parameters

show_ok("IT-05 - política NO_EFFECTIVE_CHANGES preserva is_validated")

OK - IT-05 - política NO_EFFECTIVE_CHANGES preserva is_validated


### Test IT-06: Caso de política clave adicional: strict=True escala error de operación

In [23]:
trips = tripdataset_canonical_small()
# Agrego una columna no canónica que colisionará con un target ya existente
trips_with_collision = make_tripdataset_fixture(
    pd.DataFrame({
        "movement_id": ["m1", "m2"],
        "user_id": ["u1", "u2"],
        "mode": ["bus", "walk"],
        "modo_raw": ["BUS", "WALK"],
        "purpose": ["work", "study"],
        "trip_weight": [1.0, 2.0],
    }),
    dataset_id="tripdataset_strict_collision",
    is_validated=True,
)

raised = None
try:
    fix_trips_correspondence(
        trips_with_collision,
        field_corrections={"modo_raw": "mode"},  # target ya existe
        value_corrections=None,
        options=FixCorrespondenceOptions(strict=True),
        correspondence_context={"reason": "test strict"},
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, FixError)

display(raised.issues)
show_ok("IT-06 - strict=True escala error de operación")

(Issue(level='error', code='FIX.FIELD.TARGET_ALREADY_EXISTS', message="No se puede aplicar la corrección de campo 'modo_raw' -> 'mode' porque el target ya existe en el dataset y la regla implicaría sobrescritura o colisión.", field='mode', source_field='modo_raw', row_count=None, details={'kind': 'field_corrections', 'n_rows_total': 2, 'sample_rows_per_issue': 50, 'source_column': 'modo_raw', 'target_column': 'mode', 'reason': 'target_already_exists_in_dataset', 'action': 'skip_rule'}),
 Issue(level='warning', code='FIX.NO_EFFECTIVE_CHANGES.NO_EFFECTIVE_CHANGES', message='Se proporcionaron correcciones, pero la operación terminó sin cambios efectivos en el dataset.', field=None, source_field=None, row_count=None, details={'kind': 'noop', 'field_corrections_provided': True, 'value_corrections_provided': False, 'requested_field_rules': 1, 'requested_value_rules': 0, 'n_rows_total': 2}))

OK - IT-06 - strict=True escala error de operación
